In [1]:
from PIL import Image, ExifTags
import cv2
import numpy as np
import os
import pandas as pd
import csv

CANVIA EL NOM DE LA IMATGE AL PATH SENSE FER RESET DE LA SESSIÓ!!! NO HAURIA D'HAVER CAP PROBLEMA

In [5]:

#load the image
image_path = "C:/Users/janaz/Documents/uni/YEAR 3 - S2/synthesis project/4355/HM20241217191607.jpeg" #CHANGE THIS PATH IF NEEDED
original_filename = os.path.basename(image_path)  

# open the image using PIL to access metadata
pil_image = Image.open(image_path)

#extract metadata (EXIF) if available to copy it later
exif_data = pil_image.info.get("exif")  

#extract metadata (EXIF) to print it
exif_extract = pil_image._getexif()
metadata = {}
if exif_extract:
    for tag, value in exif_extract.items():
        decoded = ExifTags.TAGS.get(tag, tag)
        metadata[decoded] = value

print("Metadata:")
for key, value in metadata.items():
    print(f"{key}: {value}")

#show orientation if available
orientation = metadata.get('Orientation')
if orientation:
    print(f"orientation EXIF detected: {orientation}") #if orientation == 1, then is good

pil_image = pil_image.convert("RGB") # Convert to RGB if not already in that mode so rotation is not a problem

# PIL (RGB) --> OpenCV (BGR)
opencv_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
image_copy = opencv_image.copy()

# Define categories and colors for different keypoints
categories = {
    't': 'torso',  
    'lh': 'left_hand',
    'rh': 'right_hand',  
    'lf': 'left_foot',
    'rf': 'right_foot',}
colors = {
    'torso': (0, 255, 255),   # Yellow
    'left_hand': (0, 0, 255),     # Red
    'right_hand': (0, 0, 180),     # Dark Red
    'left_foot': (0, 255, 0),     # Green
    'right_foot': (0, 180, 0)      # Dark Green
}


Metadata:
ExifOffset: 70
DateTime: 2024:12:17 19:16:07
Orientation: 1
DateTimeOriginal: 2024:12:17 19:16:07
DateTimeDigitized: 2024:12:17 19:16:07
orientation EXIF detected: 1


In [3]:
#NO ES NECESSARRI RUNEAR
#open the image using PIL to see the image 
pil_image.show()


START THE CODE:
BELOW THIS CODE THERE IS WHAT KEYPOINT YOU HAVE TO MARK

- CLICK R TO RESET THE KEYPOINTS TO DO IT AGAIN
- CLICK S TO SKIP THE CURRENT KEYPOINT IN CASE THERE ISN'T THE PART OF THE BODY 
- CLICK ESC ONCE FINISHING MARKING

CLICK ESC AGAIN TO PRINT THE KEYPOINTS 

CHANGE THE PATH TO THE NEXT IMAGE IN THE PREVIOUS CODE 

In [6]:
ordered_categories = ['torso', 'left_hand', 'right_hand', 'left_foot', 'right_foot']
keypoints = {cat: [] for cat in ordered_categories}
current_index = 0  

#to put the keypoints in the image
def click_event(event, x, y, flags, params):
    global image_copy, current_index

    if event == cv2.EVENT_LBUTTONDOWN:
        if current_index < len(ordered_categories):
            category = ordered_categories[current_index]
            keypoints[category].append((x, y))

            #drowing keypoints on the image
            cv2.circle(image_copy, (x, y), 5, colors.get(category, (255, 255, 255)), -1)
            cv2.putText(image_copy, category, (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors.get(category, (255, 255, 255)), 1)
            
            #show the updated image with the keypoints marked
            cv2.imshow('Image', image_copy)

            print(f"{category} marked in: ({x}, {y})")

            #next category
            current_index += 1

            if current_index < len(ordered_categories):
                print(f"next category: {ordered_categories[current_index]}")
            else:
                print("All keypoints marked. Press 'r' to reset or ESC to exit.")

def reset_annotations():
    global keypoints, image_copy, current_index
    keypoints = {cat: [] for cat in ordered_categories}
    image_copy = opencv_image.copy()
    current_index = 0
    print("Complete reset")
    print(f"Actual category: {ordered_categories[current_index]}")
    cv2.imshow('Image', image_copy)

#original image
cv2.imshow('Image', image_copy)
cv2.setMouseCallback('Image', click_event)
print(f"Initial keypoint: {ordered_categories[0]}")

while True:
    key = cv2.waitKey(0) & 0xFF

    if key == 27:  #esc to exit/end the marking
        break
    elif key == ord('r'):  # 'r' to reset annotations
        reset_annotations()
    elif key == ord('s') and current_index < len(ordered_categories): # 's' to skip the current category, it sets the keypoint to None
        category = ordered_categories[current_index]
        keypoints[category].append((None, None))
        print(f"Skipping {category}.")
        current_index += 1
        if current_index < len(ordered_categories):
            print(f"next category: {ordered_categories[current_index]}")
        else:
            print("all keypoints marked")

cv2.destroyAllWindows()

#shows the final image with all the keypoints marked
final_image = opencv_image.copy()
for category, points in keypoints.items():
    for (x, y) in points:
        if x is not None and y is not None:
            cv2.circle(final_image, (x, y), 5, colors.get(category, (255, 255, 255)), -1)
            cv2.putText(final_image, category, (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors.get(category, (255, 255, 255)), 1)

cv2.imshow('Final Annotated Image', final_image)

#convert the final image to RGB for saving with the original metadata using PIL
final_rgb = cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB)
final_pil = Image.fromarray(final_rgb)
new_filename = "val_" + original_filename         
#intoduce the path where you want to save the image
output_folder = "C:/Users/janaz/Documents/uni/YEAR 3 - S2/synthesis project/img_val_points"  #CHANGE THIS PATH IF NEEDED
output_filename = os.path.join(output_folder, new_filename)

os.makedirs(output_folder, exist_ok=True)

final_pil.save(output_filename, format="JPEG", exif=exif_data)

cv2.waitKey(0)
cv2.destroyAllWindows()

#show the keypoints
print("Keypoints:", keypoints)



Initial keypoint: torso
torso marked in: (273, 299)
next category: left_hand
Skipping left_hand.
next category: right_hand
right_hand marked in: (199, 73)
next category: left_foot
Keypoints: {'torso': [(273, 299)], 'left_hand': [(None, None)], 'right_hand': [(199, 73)], 'left_foot': [], 'right_foot': []}


SAVE KEYPOINTS AND TEMPERATURES IN A CSV FILE

In [4]:
#ONCE YOU HAVE MARKED THE KEYPOINTS, THIS PART OF THE CODE WILL SAVE THEM IN A CSV FILE

csv_path = "C:/Users/janaz/Documents/uni/YEAR 3 - S2/synthesis project/output_keypoints.csv"   #CHANGE THIS PATH IF NEEDED

header = [
    "Image Name", "Max Temp", "Min Temp",
    "torso_x", "torso_y",
    "left_hand_x", "left_hand_y",
    "right_hand_x", "right_hand_y",
    "left_foot_x", "left_foot_y",
    "right_foot_x", "right_foot_y"
]


def get_point_values(point_list):
    if point_list and point_list[0] != (None, None):
        return [point_list[0][0], point_list[0][1]]
    else:
        return [None, None]
    
row = [original_filename, "", ""]  # Max Temp y Min Temp vacíos
for part in ["torso", "left_hand", "right_hand", "left_foot", "right_foot"]:
    row.extend(get_point_values(keypoints.get(part, [(None, None)])))

#checks if the CSV file already exists and writes the information using appending mode
file_exists = os.path.isfile(csv_path)

with open(csv_path, mode='a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    #first time it runs, write the header
    if not file_exists:
        writer.writerow(header)
    
    #adds a new row with the keypoints
    writer.writerow(row)

print("information saved in CSV")

information saved in CSV


VERIFICATION OF METADATA OF THE NEW IMAGE WITH MARKED KEYPOINTS

In [ ]:
 
image_path = "C:/Users/janaz/Documents/uni/YEAR 3 - S2/synthesis project/img_val_points/val_HM20241211075636.jpeg" #CHANGE THIS PATH IF NEEDED
original_filename = os.path.basename(image_path)  
# open the image using PIL to access metadata
pil_image = Image.open(image_path)
# extract metadata (EXIF) if available
exif_extract = pil_image._getexif()
metadata = {}
if exif_extract:
    for tag, value in exif_extract.items():
        decoded = ExifTags.TAGS.get(tag, tag)
        metadata[decoded] = value

# metadata
print("Metadata:")
for key, value in metadata.items():
    print(f"{key}: {value}")


Metadata:
ExifOffset: 70
DateTime: 2024:12:11 07:56:36
Orientation: 1
DateTimeOriginal: 2024:12:11 07:56:36
DateTimeDigitized: 2024:12:11 07:56:36
